# Bloom — Kaggle GPU training for acne detection

This notebook trains and evaluates the one-class acne detector on the prepared `acne_mvp` dataset.

Before running:
1. In Kaggle, create a **private Dataset** by uploading the local `ml/data/acne_mvp` folder.
2. Attach that dataset through **Add Input**.
3. Open Notebook **Settings** and choose **Accelerator → GPU**. Turn Internet on for the first run so packages and pretrained weights can download.
4. Run all cells in order. The final bundle will be placed in `/kaggle/working/bloom_acne_artifacts.zip`.

> This is an educational/research model, not a medical diagnostic device.

In [ ]:
# Install only the training dependency. Kaggle already includes PyTorch with CUDA support.
%pip install -q ultralytics==8.2.18 pyyaml


In [ ]:
from pathlib import Path
import json
import shutil
from collections import Counter

import torch
import yaml
from ultralytics import YOLO

assert torch.cuda.is_available(), (
    'No GPU found. In the Kaggle right-side Settings panel, set Accelerator to GPU, then restart the session.'
)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)


## Locate the attached dataset

The cell searches the Kaggle inputs for a YOLO `classes.yaml` with `images/` and `labels/` folders. It removes the need to know your Kaggle dataset slug.

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
candidates = sorted(
    yaml_file.parent
    for yaml_file in INPUT_ROOT.rglob('classes.yaml')
    if (yaml_file.parent / 'images').is_dir() and (yaml_file.parent / 'labels').is_dir()
)

if not candidates:
    raise FileNotFoundError(
        'Could not find acne_mvp. Attach the Kaggle Dataset containing the acne_mvp folder via Add Input.'
    )
if len(candidates) > 1:
    print('Found multiple YOLO datasets:')
    for index, candidate in enumerate(candidates):
        print(f'  [{index}] {candidate}')
    raise RuntimeError('Attach only the acne_mvp dataset, then run this cell again.')

SOURCE = candidates[0]
WORK = Path('/kaggle/working/acne_mvp')
shutil.copytree(SOURCE, WORK, dirs_exist_ok=True)  # Kaggle inputs are read-only
print('Using dataset:', SOURCE)
print('Working copy:', WORK)


In [ ]:
# Validate expected split layout before spending GPU time.
split_counts = {}
for split in ('train', 'val', 'test'):
    images = list((WORK / 'images' / split).glob('*'))
    labels = list((WORK / 'labels' / split).glob('*.txt'))
    split_counts[split] = {'images': len(images), 'labels': len(labels)}
    assert images, f'No images in {split} split'
    assert len(images) == len(labels), f'Image/label mismatch in {split}: {len(images)} vs {len(labels)}'

print(json.dumps(split_counts, indent=2))

# classes.yaml contains a Windows source path. Rewrite only its path for Kaggle.
config = yaml.safe_load((WORK / 'classes.yaml').read_text(encoding='utf-8'))
config['path'] = str(WORK)
DATA_YAML = Path('/kaggle/working/acne_classes_kaggle.yaml')
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print(DATA_YAML.read_text())


## Train

`yolov8n` is intentionally used first: it is quick, GPU-friendly, and a reliable baseline. Increase `EPOCHS` after this baseline succeeds.

In [ ]:
EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 16
RUNS_DIR = Path('/kaggle/working/runs')
RUN_NAME = 'acne_yolov8n_50e'

model = YOLO('yolov8n.pt')
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,
    workers=2,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    patience=15,
    mosaic=0.0,
    mixup=0.0,
    seed=42,
    deterministic=True,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
)


## Evaluate only on the held-out test split

Do not use the training or validation metric as the final result. For detection, report mAP, precision, and recall instead of classification accuracy.

In [ ]:
RUN_DIR = RUNS_DIR / RUN_NAME
BEST_MODEL = RUN_DIR / 'weights' / 'best.pt'
assert BEST_MODEL.exists(), f'No best checkpoint found at {BEST_MODEL}'

best_model = YOLO(str(BEST_MODEL))
metrics = best_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,
    workers=2,
    project=str(RUN_DIR),
    name='test_evaluation',
    exist_ok=True,
    plots=True,
)

test_results = {
    'split': 'test',
    'precision': round(float(metrics.box.mp), 4),
    'recall': round(float(metrics.box.mr), 4),
    'map50': round(float(metrics.box.map50), 4),
    'map50_95': round(float(metrics.box.map), 4),
}
(RUN_DIR / 'test_metrics.json').write_text(json.dumps(test_results, indent=2), encoding='utf-8')
print(json.dumps(test_results, indent=2))


In [ ]:
# Save a compact, downloadable bundle in Kaggle Output.
ARTIFACT_DIR = Path('/kaggle/working/bloom_acne_artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)
shutil.copy2(BEST_MODEL, ARTIFACT_DIR / 'best.pt')
shutil.copy2(RUN_DIR / 'test_metrics.json', ARTIFACT_DIR / 'test_metrics.json')
for filename in ('results.csv', 'results.png', 'args.yaml', 'labels.jpg'):
    source = RUN_DIR / filename
    if source.exists():
        shutil.copy2(source, ARTIFACT_DIR / filename)

bundle = shutil.make_archive('/kaggle/working/bloom_acne_artifacts', 'zip', ARTIFACT_DIR)
print('Download after Save Version:')
print(bundle)
print('Best model:', ARTIFACT_DIR / 'best.pt')
